# From Reflection to Tool Use to Multi-Agent Orchestration
**Companion notebook to the Medium article** · All examples run in **offline mode by default**, without API keys. Optional live model and arXiv modes are clearly marked.

This is an educational architecture, not a validated scientific literature review.

## 0 · Configuration
Run the cells in order. **OFFLINE=True** ensures a no-key, no-network demonstration. To use a live LLM, set `OFFLINE=False`, install `openai`, and set `OPENAI_API_KEY` in your environment (never paste a key into a notebook).

In [ ]:
import os, json, re, time, urllib.parse, urllib.request, xml.etree.ElementTree as ET
from pprint import pprint

OFFLINE = True
MODEL = "gpt-4o-mini"  # Example API model; choose an accessible model in your account.

def model_text(messages, *, offline_fallback):
    """Return sample text offline or call the OpenAI Chat Completions API."""
    if OFFLINE:
        return offline_fallback
    from openai import OpenAI
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY in your environment.")
    client = OpenAI()
    answer = client.chat.completions.create(model=MODEL, messages=messages)
    return answer.choices[0].message.content or ""

print("Mode:", "offline demonstration" if OFFLINE else "live API")

## 1 · Reflection: draft → critique → revision
Here each step has one responsibility. Offline output is deliberately simple and explicitly illustrative.

In [ ]:
def generate_draft(topic):
    prompt = f"Write a balanced, accessible paragraph about: {topic}. Include a limitation."
    return model_text([{"role":"user", "content":prompt}],
        offline_fallback=f"{topic} is an important research question. Methods can be useful, but results depend on data quality and validation.")

def reflect_on_draft(draft):
    prompt = "Critique this draft for clarity, evidence, and limitations. Do not rewrite it:\n" + draft
    return model_text([{"role":"user", "content":prompt}],
        offline_fallback="Demo critique: Explain scope, give a concrete example, and avoid unsupported generalizations.")

def revise_draft(draft, feedback):
    prompt = f"Revise this draft using the critique. Do not invent references.\nDraft:\n{draft}\nCritique:\n{feedback}"
    return model_text([{"role":"user", "content":prompt}],
        offline_fallback=draft + " A useful next step is evaluation on independent cohorts and checking whether conclusions generalize.")

topic = "Polygenic risk score prediction"
draft = generate_draft(topic)
critique = reflect_on_draft(draft)
revision = revise_draft(draft, critique)
for title, result in [("DRAFT", draft), ("CRITIQUE", critique), ("REVISION", revision)]:
    print(f"\n{title}\n{result}")

## 2 · Tool use: a paper-search function
**OFFLINE=True:** returns fictional *demonstration-only* records. They are not citations. **OFFLINE=False:** queries the public arXiv API; network availability and coverage vary. Verify every source before citing it.

In [ ]:
def search_papers(query, max_results=3):
    max_results = max(1, min(int(max_results), 5))
    if OFFLINE:
        return [{"title":"DEMO RECORD — NOT A REAL PAPER", "url":"", "summary":f"Example only; search query was: {query}", "source":"offline mock"}]
    params = urllib.parse.urlencode({"search_query":"all:" + query, "start":0, "max_results":max_results})
    req = urllib.request.Request("https://export.arxiv.org/api/query?"+params, headers={"User-Agent":"ResearchWorkflowTutorial/1.0"})
    with urllib.request.urlopen(req, timeout=15) as r:
        xml = r.read()
    root = ET.fromstring(xml)
    ns = {"a":"http://www.w3.org/2005/Atom"}
    out=[]
    for entry in root.findall("a:entry", ns):
        get=lambda tag: " ".join((entry.findtext("a:"+tag, default="", namespaces=ns) or "").split())
        out.append({"title":get("title"), "url":get("id"), "summary":get("summary")[:650], "source":"arXiv API"})
    return out

papers = search_papers("polygenic risk scores")
pprint(papers)
assert isinstance(papers, list) and all("title" in p for p in papers)

## 3 · Model tool calling (optional live demonstration)
A model's tool request is **not** a Python execution. This helper dispatches only one allowlisted function, validates JSON arguments, associates the result with `tool_call_id`, and stops after a bounded number of rounds. Offline mode shows the tool manually instead.

In [ ]:
TOOL_SPEC = [{"type":"function", "function":{
    "name":"search_papers", "description":"Search arXiv for scientific papers by topic",
    "parameters":{"type":"object", "properties":{"query":{"type":"string"}},
                  "required":["query"], "additionalProperties":False}
}}]

def answer_with_tools(question, max_rounds=3):
    if OFFLINE:
        found=search_papers(question)
        return {"mode":"offline demo", "question":question, "tool_results":found,
                "answer":"Illustrative answer only. Run in live mode for real retrieval and synthesis."}
    from openai import OpenAI
    client=OpenAI()
    messages=[{"role":"system", "content":"Use available research tools where helpful. Do not invent citations."},
              {"role":"user", "content":question}]
    for _ in range(max_rounds):
        response=client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPEC, tool_choice="auto")
        msg=response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return {"mode":"live", "answer":msg.content or "", "messages":messages}
        for call in msg.tool_calls:
            if call.function.name != "search_papers":
                result={"error":"Tool not permitted"}
            else:
                try:
                    args=json.loads(call.function.arguments)
                    if set(args) != {"query"} or not isinstance(args["query"], str) or len(args["query"]) > 250:
                        raise ValueError("Expected one string query (max 250 characters)")
                    result=search_papers(args["query"])
                except Exception as exc:
                    result={"error":str(exc)}
            messages.append({"role":"tool", "tool_call_id":call.id, "content":json.dumps(result)})
    return {"mode":"live", "answer":"Stopped at maximum tool rounds", "messages":messages}

pprint(answer_with_tools("Find research about PRS cross-population transferability"))

## 4 · Multi-agent orchestration
Use a bounded plan with explicit data flow: plan → retrieve → write → edit → revise. The offline demonstration illustrates wiring rather than scientific accuracy.

In [ ]:
def planner_agent(question):
    prompt=f"Plan at most 3 concise research tasks for: {question}. Return only numbered lines."
    text=model_text([{"role":"user", "content":prompt}], offline_fallback="1. Search relevant research.\n2. Summarize methods and caveats.\n3. Critically review the summary.")
    lines=[re.sub(r"^\s*\d+[.)]\s*", "", x).strip() for x in text.splitlines() if x.strip()]
    return lines[:3]

def research_agent(question):
    return search_papers(question, max_results=3)

def writer_agent(question, evidence):
    snippets=json.dumps(evidence, ensure_ascii=False)[:6000]
    prompt=f"Write a short evidence-aware research overview for {question}. Use only these retrieved notes and admit gaps:\n{snippets}"
    fallback=f"Research overview (DEMO): {question}. Retrieved {len(evidence)} demonstration record(s); no scientific conclusions can be justified from mock records."
    return model_text([{"role":"user", "content":prompt}], offline_fallback=fallback)

def editor_agent(draft, evidence):
    prompt=f"Critique this draft against the supplied sources, distinguishing supported and unsupported claims:\n{draft}\nSources:\n{json.dumps(evidence)[:6000]}"
    return model_text([{"role":"user", "content":prompt}], offline_fallback="DEMO review: Do not cite sample records as real evidence. Verify all source claims.")

def executor_agent(question):
    plan=planner_agent(question)
    evidence=research_agent(question)
    draft=writer_agent(question, evidence)
    review=editor_agent(draft, evidence)
    final=revise_draft(draft, review)
    return {"question":question, "plan":plan, "evidence":evidence,
            "draft":draft, "editor_feedback":review, "final_report":final}

result=executor_agent("Challenges in PRS transferability across populations")
for name, value in result.items():
    print("\n###", name.upper())
    pprint(value)
assert set(result) == {"question", "plan", "evidence", "draft", "editor_feedback", "final_report"}

## 5 · Exercises to go further
1. Save the evidence as JSON and record timestamps of retrieval.
2. Add DOI/arXiv ID checking before accepting a citation.
3. Create an evaluation rubric for citation fidelity, clarity, and limitations.
4. Compare two versions of a report using independent assessment, not just self-critique.
5. Add retry and rate-limit handling for network and API failures.

**Publishing reminder:** offline example records are fabricated demo data, never real scientific evidence. Run and verify live retrieval before claiming a literature review has been performed.